In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer

c:\Workspace\MyProjects\DeepLearning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1. Load your CSV data
csv_file_path = 'tickets.csv' # Make sure this path is correct
df = pd.read_csv(csv_file_path)

In [5]:
df.head()

,LogID,Symptom,Issue,Resolution
0,113551,Version # (if applicable):34978 IsScrapBin (pl...,Run solve having error,"HXVaf have nsmusd for 2 conversion groups, whi..."
1,113565,Version # (if applicable): WW25 version IsScra...,OneMPS does not have RTF for 2025Q1 for all so...,RC: extra 2025Q1 quarter in weekly version du...
2,113567,Version # (if applicable):34994 IsScrapBin (pl...,Error to pull Limiter Chart By VGG report. FVL...,The issue has been fixed. Pls help to verify -
3,113569,Version # (if applicable): Data Staging IsScra...,"can't save "" modify conversion group'",Explaination provided: Need to save record fir...
4,113570,Version # (if applicable): Data staging.. IsSc...,NaN,HX_CT_MB_ARROW_LAKE_H_6C+8A+GT2_N3B_HFE is co...


In [4]:
df.drop(['LogID', 'Symptom'], axis='columns', inplace=True)
df.head()

,Issue,Resolution
0,Run solve having error,"HXVaf have nsmusd for 2 conversion groups, whi..."
1,OneMPS does not have RTF for 2025Q1 for all so...,RC: extra 2025Q1 quarter in weekly version du...
2,Error to pull Limiter Chart By VGG report. FVL...,The issue has been fixed. Pls help to verify -
3,"can't save "" modify conversion group'",Explaination provided: Need to save record fir...
4,NaN,HX_CT_MB_ARROW_LAKE_H_6C+8A+GT2_N3B_HFE is co...


In [37]:
df.tail()

,Issue,Resolution
388,I was trying to publish Site Response with ATC...,I have already pulled the data from the publis...
389,When I try to download the planning management...,Time out issue when we select all the locatio...
390,I run solve ST6 to APN410 Assy-Test to see max...,NaN
391,Run solve fail. Data fetch error on CapacityIt...,VG mapped to Dummy and non-dummy request item...
392,I am unable to connect to VPN,"Restart the PC, connect GlobalProtect connect,..."


In [8]:
# Filter out rows where 'Issue' is NaN
issues = df['Issue'].dropna().tolist()

In [10]:
# 2. Initialize the SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:

issue_embeddings = model.encode(issues, convert_to_tensor=True)

KeyboardInterrupt: 

In [13]:
import faiss
import numpy as np

# Convert embeddings to a NumPy array (FAISS expects NumPy arrays)
issue_embeddings_np = issue_embeddings.cpu().numpy()

# Get the dimensionality of the embeddings
embedding_dimension = issue_embeddings_np.shape[1]

In [16]:
# Create a FAISS index
# IndexFlatL2 is a simple index that performs brute-force L2 (Euclidean) distance search.
# For larger datasets, you might consider IndexIVFFlat or others for faster approximate nearest neighbor search.
index = faiss.IndexFlatL2(embedding_dimension)

# Add the embeddings to the index
index.add(issue_embeddings_np)
print(f"Number of vectors in the FAISS index: {index.ntotal}")

Number of vectors in the FAISS index: 334


In [17]:
# Optional: Save the FAISS index (for persistence)
faiss.write_index(index, "issue_resolution_faiss_index.bin")

In [ ]:
# Optional: To load the index later:
# loaded_index = faiss.read_index("issue_resolution_faiss_index.bin")

In [18]:
# Function to query the vector store
def find_similar_issues(query_text, model, faiss_index, df, k=3):
    """
    Finds top k similar issues in the FAISS index for a given query text.

    Args:
        query_text (str): The problem statement to search for.
        model (SentenceTransformer): The embedding model.
        faiss_index (faiss.Index): The FAISS index containing issue embeddings.
        df (pd.DataFrame): The original DataFrame with 'Issue' and 'Resolution' columns.
        k (int): The number of top similar results to retrieve.

    Returns:
        pd.DataFrame: A DataFrame containing the top k similar issues and their resolutions,
                      along with their similarity scores (Euclidean distance).
    """
    # Generate embedding for the query
    query_embedding = model.encode([query_text], convert_to_tensor=True).cpu().numpy()

    # Perform similarity search
    # D: distances (L2 distances in this case, lower is more similar)
    # I: indices of the nearest neighbors in the FAISS index
    distances, indices = faiss_index.search(query_embedding, k)

    # Prepare results
    results = []
    for i in range(k):
        original_index = indices[0][i]
        distance = distances[0][i]
        results.append({
            "Query": query_text,
            "Similar Issue": df.loc[original_index, 'Issue'],
            "Proposed Resolution": df.loc[original_index, 'Resolution'],
            "Distance (L2)": distance
        })
    return pd.DataFrame(results)

In [ ]:
# Example Usage:
new_problem = "Unable to connect to the VPN"
similar_results = find_similar_issues(new_problem, model, index, df, k=10)

# convert the similar_results to dataframe
pd.DataFrame(similar_results).head()


,Query,Similar Issue,Proposed Resolution,Distance (L2)
0,Unable to connect to the VPN,"Run solve version, then copy Cap Used VGG to X...",APN41 test is strategy 5 and cap target zero....,0.097226
1,Unable to connect to the VPN,W1 + W2 site response is less than CC and over...,"For CTO finish prod, there is a rule that only...",1.279426
2,Unable to connect to the VPN,Need help to check on BOH data for FALCON MESA...,NaN,1.376590
3,Unable to connect to the VPN,min allocation subject to change although is i...,Frozen horizon is 2 weeks in that version. Use...,1.418174
4,Unable to connect to the VPN,1. Checking is not constrain at HDM_AP_DKM_ARL...,There were some mappings missing for the menti...,1.458709


In [ ]:
from sentence_transformers import util
model = SentenceTransformer('all-mpnet-base-v2') # Or another model

# Generate embeddings
issue_embeddings = model.encode(df['Issue'].dropna().tolist(), convert_to_tensor=True)

# NORMALIZE EMBEDDINGS TO UNIT VECTORS
issue_embeddings_normalized = util.normalize_embeddings(issue_embeddings).cpu().numpy()

embedding_dimension = issue_embeddings_normalized.shape[1]

# Create a FAISS index (still IndexFlatL2, but now on normalized vectors)
index = faiss.IndexFlatL2(embedding_dimension)
index.add(issue_embeddings_normalized)

print(f"Number of vectors in the FAISS index: {index.ntotal}")

# --- When querying ---
def find_similar_issues_normalized(query_text, model, faiss_index, df, k=3):
    query_embedding = model.encode([query_text], convert_to_tensor=True)
    # NORMALIZE THE QUERY EMBEDDING AS WELL
    query_embedding_normalized = util.normalize_embeddings(query_embedding).cpu().numpy()

    distances, indices = faiss_index.search(query_embedding_normalized, k)

    results = []
    for i in range(k):
        original_index = indices[0][i]
        # When embeddings are L2-normalized, L2 distance (d) is related to cosine similarity (cos_sim) by:
        # d^2 = 2 - 2 * cos_sim
        # So, cos_sim = 1 - (d^2 / 2)
        distance = distances[0][i]
        cosine_similarity = 1 - (distance**2 / 2) # Calculate cosine similarity for interpretability

        results.append({
            "Query": query_text,
            "Similar Issue": df.loc[original_index, 'Issue'],
            "Proposed Resolution": df.loc[original_index, 'Resolution'],
            "L2 Distance": distance,
            "Cosine Similarity": cosine_similarity # Display cosine similarity
        })
    return pd.DataFrame(results)



In [36]:
# Test with normalized query
new_problem = "I am unable to connect to VPN"
similar_results = find_similar_issues_normalized(new_problem, model, index, df, k=2)

pd.DataFrame(similar_results).head()

,Query,Similar Issue,Proposed Resolution,L2 Distance,Cosine Similarity
0,I am unable to connect to VPN,"Run solve version, then copy Cap Used VGG to X...",APN41 test is strategy 5 and cap target zero....,1.131702e-12,1.000000
1,I am unable to connect to VPN,W1 + W2 site response is less than CC and over...,"For CTO finish prod, there is a rule that only...",1.093986e+00,0.401598


In [5]:
model = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
model.eval()

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [9]:
import torch
with torch.no_grad(): # Disable gradient calculation for inference
    issue_embeddings = model.encode(issues, convert_to_tensor=True, show_progress_bar=True, batch_size=32)


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches: 100%|██████████| 11/11 [02:05<00:00, 11.42s/it]


In [10]:
issues[0:2], issue_embeddings[0:2]

(['Run solve having error',
  'OneMPS does not have RTF for 2025Q1 for all solve groups. The bucket Range is 202424 to 202513'],
 tensor([[ 0.0184,  0.2524,  0.0160,  ..., -0.1866, -0.1622,  0.1512],
         [ 0.2841,  0.1185, -0.2703,  ..., -0.5721, -0.2207, -0.0067]]))

In [37]:
df.tail()

,Issue,Resolution
388,I was trying to publish Site Response with ATC...,I have already pulled the data from the publis...
389,When I try to download the planning management...,Time out issue when we select all the locatio...
390,I run solve ST6 to APN410 Assy-Test to see max...,NaN
391,Run solve fail. Data fetch error on CapacityIt...,VG mapped to Dummy and non-dummy request item...
392,I am unable to connect to VPN,"Restart the PC, connect GlobalProtect connect,..."


In [52]:
df.dropna(subset=['Issue'], axis=0, inplace=True)
issues = df['Issue'].tolist()
resolutions = df['Resolution'].tolist()
issue_ids = df.index.tolist()

In [58]:
len(issue_ids)

334

In [51]:
df.tail()

,Issue,Resolution
388,I was trying to publish Site Response with ATC...,I have already pulled the data from the publis...
389,When I try to download the planning management...,Time out issue when we select all the locatio...
390,I run solve ST6 to APN410 Assy-Test to see max...,NaN
391,Run solve fail. Data fetch error on CapacityIt...,VG mapped to Dummy and non-dummy request item...
392,I am unable to connect to VPN,"Restart the PC, connect GlobalProtect connect,..."


In [13]:
from sentence_transformers import util

In [66]:
def find_similar_issues(query_text: str, k: int = 10):
    """
    Finds the top 3 most semantically similar issues to a given query text.

    Args:
        query_text (str): The text query for which to find similar issues.
        k (int): The number of top similar issues to retrieve.

    Returns:
        list: A list of dictionaries, each containing 'id', 'score', and 'issue_text'.
    """
    print(f"\n--- Searching for similar issues to: '{query_text}' ---")

    # Generate embedding for the query
    with torch.no_grad():
        query_embedding = model.encode(query_text, convert_to_tensor=True)
        if torch.cuda.is_available():
            query_embedding = query_embedding.to('cuda')

    # Calculate cosine similarity between the query and all existing issue embeddings
    # util.cos_sim returns a similarity matrix. We take the first (and only) row.
    cosine_scores = util.cos_sim(query_embedding, issue_embeddings)[0]

    # Get the top_k indices and their scores
    # top_results[0] are the scores, top_results[1] are the indices
    top_results = torch.topk(cosine_scores, k=k)

    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        if score.item() > 0.6:
            result = {
                'query': query_text,
                'issue': issues[idx],
                'score': score.item(), # .item() converts tensor to Python number
                'resolution': resolutions[idx]
            }
            results.append(result)

    return results


In [60]:
df.loc[333]

Issue         Run solve version, then copy Cap Used VGG to X...
Resolution     APN41 test is strategy 5 and cap target zero....
Name: 333, dtype: object

In [67]:
# Example 1: VPN related queries
result = find_similar_issues("My VPN isn't working at all.")
pd.DataFrame(result).head()


--- Searching for similar issues to: 'My VPN isn't working at all.' ---


,query,issue,score,resolution
0,My VPN isn't working at all.,I am unable to connect to VPN,0.8558,"Restart the PC, connect GlobalProtect connect,..."


In [1]:
# Example 2: VPN related queries
result = find_similar_issues("Data refresh for FH horizon is showing failure")
pd.DataFrame(result).head()

NameError: name 'find_similar_issues' is not defined